# Problem 2 — Transformer from Scratch
**DNN Programming Assignment | Implementation and Experiments**

Run from the `problem2/` folder, top to bottom (Restart & Run All). The HAR data is read from `../problem1/data/har/UCI HAR Dataset`; Tiny Shakespeare is downloaded into `data/`.

All numbers in this notebook are produced by the cells below. Compute budgets are reduced for CPU execution and are stated next to each experiment.

## 0. Setup and reproducibility

In [ ]:
import math, random, time, urllib.request
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
set_seed()
torch.set_num_threads(6)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path("."); DATA = ROOT/"data"; FIG = ROOT/"figures"
DATA.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
print("Device:", DEVICE, "PyTorch:", torch.__version__)

## 1. Task 1 — Scaled dot-product attention

For queries Q, keys K and values V, attention scores are `QKᵀ/√dₖ`. Softmax converts scores into weights; the weighted sum of V is the context. The mask convention below is **True = allowed**.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout_p=0.0, training=False):
    # Q: (..., Lq, D), K: (..., Lk, D), V: (..., Lk, Dv)
    d_k=Q.shape[-1]
    scores=torch.matmul(Q,K.transpose(-2,-1))/math.sqrt(d_k)
    if mask is not None:
        scores=scores.masked_fill(~mask.bool(),torch.finfo(scores.dtype).min)
    weights=torch.softmax(scores,dim=-1)
    if dropout_p and training: weights=F.dropout(weights,p=dropout_p)
    context=torch.matmul(weights,V)
    return context,weights

# Numerical verification against PyTorch's reference operation
q=torch.randn(2,3,5,8); k=torch.randn(2,3,7,8); v=torch.randn(2,3,7,6)
out,w=scaled_dot_product_attention(q,k,v)
ref=F.scaled_dot_product_attention(q,k,v)
assert torch.allclose(out,ref,atol=1e-5)
assert torch.allclose(w.sum(-1),torch.ones_like(w.sum(-1)),atol=1e-6)
print("Reference comparison and row-sum checks passed.")

In [ ]:
# Causal mask: each position can attend to itself and earlier positions.
L=5
causal=torch.tril(torch.ones(L,L,dtype=torch.bool))
q=k=v=torch.randn(1,1,L,8)
_,cw=scaled_dot_product_attention(q,k,v,mask=causal)
assert torch.all(cw.masked_select(~causal)==0)
print("Causal future-weight check passed.")

### 1.1 Hand-checkable 3-token example

The instructor's official example must be checked against these values (see README). The values below are a small causal example chosen for hand computation: with d_k = 2, the scores are `QKᵀ/√2`. The code result is compared with an independent loop implementation of the same equations.

In [ ]:
Qh = torch.tensor([[[1., 0.], [0., 1.], [1., 1.]]])   # (1, 3, 2)
Kh = Qh.clone()
Vh = torch.tensor([[[1., 2.], [3., 4.], [5., 6.]]])   # (1, 3, 2)
causal3 = torch.tril(torch.ones(3, 3, dtype=torch.bool))
ctx_h, w_h = scaled_dot_product_attention(Qh, Kh, Vh, mask=causal3)

# Independent loop implementation (one query at a time, causal prefix only)
ref = torch.zeros_like(Vh); ref_w = torch.zeros(1, 3, 3)
for i in range(3):
    s = torch.tensor([float(Qh[0, i] @ Kh[0, j]) / math.sqrt(2) for j in range(i + 1)])
    p = torch.softmax(s, 0)
    ref_w[0, i, :i + 1] = p
    ref[0, i] = (p[:, None] * Vh[0, :i + 1]).sum(0)
assert torch.allclose(ctx_h, ref, atol=1e-6) and torch.allclose(w_h, ref_w, atol=1e-6)
print("Attention weights (rows = queries):\n", w_h[0].numpy().round(4))
print("Context output:\n", ctx_h[0].numpy().round(4))
print("Hand-example code check passed.")

## 2. Task 2 — Positional encoding, multi-head attention and encoder

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self,d_model,max_len=4096):
        super().__init__()
        position=torch.arange(max_len,dtype=torch.float32).unsqueeze(1)
        div=torch.exp(torch.arange(0,d_model,2,dtype=torch.float32)*(-math.log(10000.0)/d_model))
        pe=torch.zeros(max_len,d_model)
        pe[:,0::2]=torch.sin(position*div)
        pe[:,1::2]=torch.cos(position*div[:pe[:,1::2].shape[1]])
        self.register_buffer("pe",pe.unsqueeze(0))
    def forward(self,x): return x+self.pe[:,:x.size(1)]

class MultiHeadAttention(nn.Module):
    def __init__(self,d_model,num_heads,dropout=0.1):
        super().__init__()
        if d_model%num_heads: raise ValueError("d_model must be divisible by num_heads")
        self.d_model=d_model; self.num_heads=num_heads; self.d_head=d_model//num_heads
        self.q_proj=nn.Linear(d_model,d_model); self.k_proj=nn.Linear(d_model,d_model)
        self.v_proj=nn.Linear(d_model,d_model); self.out_proj=nn.Linear(d_model,d_model)
        self.dropout=dropout
    def forward(self,x,mask=None,return_attention=False):
        B,L,D=x.shape
        def split(t): return t.view(B,L,self.num_heads,self.d_head).transpose(1,2)
        q,k,v=split(self.q_proj(x)),split(self.k_proj(x)),split(self.v_proj(x))
        context,weights=scaled_dot_product_attention(q,k,v,mask, self.dropout,self.training)
        context=context.transpose(1,2).contiguous().view(B,L,D)
        result=self.out_proj(context)
        return (result,weights) if return_attention else result

class EncoderBlock(nn.Module):
    def __init__(self,d_model=128,num_heads=4,d_ff=512,dropout=0.1):
        super().__init__()
        self.norm1=nn.LayerNorm(d_model); self.attn=MultiHeadAttention(d_model,num_heads,dropout)
        self.norm2=nn.LayerNorm(d_model)
        self.ff=nn.Sequential(nn.Linear(d_model,d_ff),nn.GELU(),nn.Dropout(dropout),
                              nn.Linear(d_ff,d_model),nn.Dropout(dropout))
    def forward(self,x,mask=None):
        x=x+self.attn(self.norm1(x),mask)
        x=x+self.ff(self.norm2(x))
        return x

block=EncoderBlock().to(DEVICE)
print("Encoder block parameters:",sum(p.numel() for p in block.parameters()))

### 2.1 Encoder-block parameter count

Per block (d = 128, d_ff = 512):
- weights: 4d² (W_Q, W_K, W_V, W_O) + 2·d·d_ff (two FF matrices) = 65,536 + 131,072 = 196,608
- biases: 4d (attention) + (d_ff + d) (feed-forward) = 512 + 640 = 1,152
- LayerNorm (two layers, γ and β each): 2 × 2d = 512

Total = 196,608 + 1,152 + 512 = **198,272**, which matches the printed count above.

In [ ]:
d, dff = 128, 512
expected = 4*d*d + 2*d*dff + 4*d + (dff + d) + 2*2*d
assert sum(p.numel() for p in EncoderBlock(d, 4, dff).parameters()) == expected == 198272
print("Encoder-block parameter count matches the derivation:", expected)

## 3. Task A — HAR activity classification

Uses the UCI HAR inertial-signal files from Problem 1 (official train/test partitions). The validation split holds out training subjects (`GroupShuffleSplit` by subject). Normalization statistics are computed only on the training subset. The test split is the official UCI HAR test set, the same one used in Problem 1.

In [ ]:

from pathlib import Path
import numpy as np
from sklearn.model_selection import GroupShuffleSplit

# --------------------------------------------------
# 1. Locate UCI HAR Dataset
# --------------------------------------------------

candidates = [
    Path("../problem1/data/har/UCI HAR Dataset"),   # notebook runs from problem2/
    Path("problem1/data/har/UCI HAR Dataset"),      # if started from the project root
]

HAR_ROOT = next(
    (
        p.resolve()
        for p in candidates
        if (p / "train").is_dir()
        and (p / "test").is_dir()
    ),
    None
)

if HAR_ROOT is None:
    raise FileNotFoundError(
        "UCI HAR Dataset not found. "
        "Expected location: problem1/data/har/UCI HAR Dataset"
    )

print("HAR dataset found at:", HAR_ROOT)


# --------------------------------------------------
# 2. Define inertial signals
# --------------------------------------------------

SIGNALS = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]


# --------------------------------------------------
# 3. Load HAR data
# --------------------------------------------------

def load_har(split, root=HAR_ROOT):

    signal_dir = root / split / "Inertial Signals"

    arrays = [
        np.loadtxt(signal_dir / f"{s}_{split}.txt")
        for s in SIGNALS
    ]

    X = np.stack(arrays, axis=-1).astype(np.float32)

    y = np.loadtxt(
        root / split / f"y_{split}.txt",
        dtype=np.int64
    ) - 1

    subjects = np.loadtxt(
        root / split / f"subject_{split}.txt",
        dtype=np.int64
    )

    return X, y, subjects


# --------------------------------------------------
# 4. Load original training and test sets
# --------------------------------------------------

X_all, y_all, sub_all = load_har("train")
X_test, y_test, sub_test = load_har("test")

print("Original training:", X_all.shape)
print("Original test:", X_test.shape)


# --------------------------------------------------
# 5. Create subject-independent validation split
# --------------------------------------------------

SEED = 42

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

itr, iv = next(
    splitter.split(
        X_all,
        y_all,
        groups=sub_all
    )
)

Xtr, Xval = X_all[itr], X_all[iv]
ytr, yval = y_all[itr], y_all[iv]


# --------------------------------------------------
# 6. Normalize using training data only
# --------------------------------------------------

mean = Xtr.mean(axis=(0, 1), keepdims=True)
std = Xtr.std(axis=(0, 1), keepdims=True) + 1e-6

Xtr = (Xtr - mean) / std
Xval = (Xval - mean) / std
X_test = (X_test - mean) / std


# --------------------------------------------------
# 7. Verify dataset and subject split
# --------------------------------------------------

print("\nDataset successfully loaded and normalized")

print("Training:", Xtr.shape, ytr.shape)
print("Validation:", Xval.shape, yval.shape)
print("Test:", X_test.shape, y_test.shape)

print("\nTraining subjects:", np.unique(sub_all[itr]))
print("Validation subjects:", np.unique(sub_all[iv]))

assert set(sub_all[itr]).isdisjoint(set(sub_all[iv])), \
    "Subject leakage detected!"

print("\nSubject-independent split verified.")


In [ ]:
from torch.utils.data import TensorDataset, DataLoader
def har_loader(X, y, batch=128, shuffle=False):
    return DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y, dtype=torch.long)),
                      batch_size=batch, shuffle=shuffle)
train_loader = har_loader(Xtr, ytr, shuffle=True); val_loader = har_loader(Xval, yval); test_loader = har_loader(X_test, y_test)

def parameter_count(model): return sum(p.numel() for p in model.parameters() if p.requires_grad)

class LearnedPositionalEmbedding(nn.Module):
    # One trainable vector per position (CLS + 128 time steps); x: (B, L, D)
    def __init__(self, d_model, max_len):
        super().__init__(); self.emb = nn.Embedding(max_len, d_model); nn.init.normal_(self.emb.weight, std=.02)
    def forward(self, x): return x + self.emb(torch.arange(x.size(1), device=x.device))[None]

class HARTransformer(nn.Module):
    # positional: "none" | "sinusoidal" | "learned"
    def __init__(self, n_features=9, n_classes=6, d_model=128, heads=4, layers=2,
                 d_ff=512, dropout=.1, positional="sinusoidal", seq_len=128):
        super().__init__()
        if positional not in ("none", "sinusoidal", "learned"): raise ValueError(positional)
        self.embed = nn.Linear(n_features, d_model)
        self.cls = nn.Parameter(torch.zeros(1, 1, d_model))
        if positional == "sinusoidal": self.pos = SinusoidalPositionalEncoding(d_model)
        elif positional == "learned": self.pos = LearnedPositionalEmbedding(d_model, seq_len + 1)
        else: self.pos = None
        self.blocks = nn.ModuleList([EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(d_model); self.head = nn.Linear(d_model, n_classes)
        nn.init.normal_(self.cls, std=.02)
    def forward(self, x):                       # x: (B, 128, 9)
        x = self.embed(x); cls = self.cls.expand(x.size(0), -1, -1); x = torch.cat([cls, x], dim=1)
        if self.pos is not None: x = self.pos(x)
        for layer in self.blocks: x = layer(x)
        return self.head(self.norm(x[:, 0]))

def train_classifier(model, train_loader, val_loader, epochs=15, lr=1e-3):
    model.to(DEVICE); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        model.train(); total = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
            total += loss.item() * len(yb)
        model.eval(); vl = 0; pred = []; true = []
        with torch.no_grad():
            for xb, yb in val_loader:
                logits = model(xb.to(DEVICE)); vl += F.cross_entropy(logits, yb.to(DEVICE)).item() * len(yb)
                pred.extend(logits.argmax(1).cpu().numpy()); true.extend(yb.numpy())
        history["train_loss"].append(total / len(train_loader.dataset))
        history["val_loss"].append(vl / len(val_loader.dataset))
        history["val_acc"].append(accuracy_score(true, pred))
        print(f"Epoch {epoch+1:02d}/{epochs} train={history['train_loss'][-1]:.4f} val={history['val_loss'][-1]:.4f} acc={history['val_acc'][-1]:.4f}")
    return history

def evaluate_classifier(model, loader):
    model.eval(); pred = []; true = []
    with torch.no_grad():
        for xb, yb in loader: pred.extend(model(xb.to(DEVICE)).argmax(1).cpu().numpy()); true.extend(yb.numpy())
    return {"accuracy": accuracy_score(true, pred), "macro_f1": f1_score(true, pred, average="macro"),
            "predictions": np.array(pred), "targets": np.array(true)}

def run_har(positional, epochs=15, seed=SEED):
    # Same seed, same budget (15 epochs, AdamW lr 1e-3, batch 128) for every positional variant
    set_seed(seed)
    model = HARTransformer(positional=positional).to(DEVICE)
    n_params = parameter_count(model)
    t0 = time.perf_counter()
    hist = train_classifier(model, train_loader, val_loader, epochs=epochs)
    sec_per_epoch = (time.perf_counter() - t0) / epochs
    return {"model": model, "history": hist, "params": n_params, "sec_per_epoch": sec_per_epoch,
            "val": evaluate_classifier(model, val_loader), "test": evaluate_classifier(model, test_loader)}

### 3.1 Positional-encoding ablation (none / sinusoidal / learned)

In [ ]:
HAR_RUNS = {}
for pos in ["none", "sinusoidal", "learned"]:
    HAR_RUNS[pos] = run_har(pos)
    r = HAR_RUNS[pos]
    print(f"\n[{pos}] params={r['params']} sec/epoch={r['sec_per_epoch']:.2f} "
          f"val_acc={r['val']['accuracy']:.4f} test_acc={r['test']['accuracy']:.4f} test_macro_f1={r['test']['macro_f1']:.4f}")
pos_table = pd.DataFrame([{"positional": p, "params": r["params"], "sec_per_epoch": round(r["sec_per_epoch"], 2),
                           "val_acc": r["val"]["accuracy"], "test_acc": r["test"]["accuracy"],
                           "test_macro_f1": r["test"]["macro_f1"]} for p, r in HAR_RUNS.items()])
display(pos_table)

### 3.2 Default HAR model (sinusoidal PE): learning curve and per-class results

In [ ]:
d = HAR_RUNS["sinusoidal"]
plt.plot(d["history"]["train_loss"], label="train"); plt.plot(d["history"]["val_loss"], label="validation")
plt.xlabel("Epoch"); plt.ylabel("Cross-entropy"); plt.legend(); plt.tight_layout()
plt.savefig(FIG/"har_learning_curve.png", dpi=160); plt.show()
ACT = ["Walking", "Upstairs", "Downstairs", "Sitting", "Standing", "Laying"]
print("Test confusion matrix (rows = true, cols = predicted):\n", confusion_matrix(d["test"]["targets"], d["test"]["predictions"]))
print(classification_report(d["test"]["targets"], d["test"]["predictions"], target_names=ACT, digits=3))

### 3.3 Comparison with the best Problem 1 recurrent model

In [ ]:
P1_CSV = Path("../problem1/results_problem1.csv")
if P1_CSV.exists():
    p1 = pd.read_csv(P1_CSV)
    rec = p1[p1["model"].astype(str).str.contains("RNN|LSTM|GRU", regex=True) & (p1["seq_len"] == 128)]
    if len(rec):
        summary = rec.groupby("model")[["test_acc", "macro_f1", "params", "sec_per_epoch"]].mean()
        print("Problem 1 recurrent HAR models (mean over seeds, official test split):")
        display(summary)
        best = summary["test_acc"].idxmax()
        print(f"Best recurrent model: {best}  test_acc={summary.loc[best, 'test_acc']:.4f}")
    else:
        print("results_problem1.csv has no RNN/LSTM/GRU HAR rows (seq_len=128) - comparison pending.")
else:
    print(f"{P1_CSV} not found - comparison pending until Problem 1 results are written.")
print(f"HAR Transformer (sinusoidal): test_acc={d['test']['accuracy']:.4f} macro_f1={d['test']['macro_f1']:.4f} params={d['params']}")

<!--PATCH:comparison-->

## 4. Task B — Character-level language model (Tiny Shakespeare)

In [ ]:
URL="https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text_path=DATA/"tinyshakespeare.txt"
if not text_path.exists(): urllib.request.urlretrieve(URL,text_path)
text=text_path.read_text(encoding="utf-8")
chars=sorted(set(text)); assert len(chars)==65, f"Expected 65 chars, found {len(chars)}"
stoi={c:i for i,c in enumerate(chars)}; itos={i:c for c,i in stoi.items()}
encoded=torch.tensor([stoi[c] for c in text],dtype=torch.long)
cut=int(.90*len(encoded)); train_tokens=encoded[:cut]; val_tokens=encoded[cut:]
print("Characters:",len(chars),"Train tokens:",len(train_tokens),"Validation tokens:",len(val_tokens))
CONTEXT=128
def get_batch(tokens,batch_size=64,context=CONTEXT):
    starts=torch.randint(0,len(tokens)-context-1,(batch_size,))
    x=torch.stack([tokens[i:i+context] for i in starts])
    y=torch.stack([tokens[i+1:i+context+1] for i in starts])
    return x.to(DEVICE),y.to(DEVICE)
def make_eval_batches(tokens, context=CONTEXT, n_batches=100, batch_size=64, seed=0):
    # Fixed validation windows: identical for every model so that losses are directly comparable
    g = torch.Generator().manual_seed(seed); out = []
    for _ in range(n_batches):
        starts = torch.randint(0, len(tokens) - context - 1, (batch_size,), generator=g)
        x = torch.stack([tokens[i:i+context] for i in starts]); y = torch.stack([tokens[i+1:i+context+1] for i in starts])
        out.append((x, y))
    return out
VAL_EVAL = make_eval_batches(val_tokens)
print("Fixed validation windows:", len(VAL_EVAL), "batches of", VAL_EVAL[0][0].shape)

### 4.1 Model

Pre-LayerNorm decoder-style stack built from the notebook's own attention and encoder block. The causal mask is `(L, L)` lower-triangular: position t may attend to positions ≤ t. The output embedding is tied to the input embedding. The embedding is initialised with std 0.02 (GPT-style); the earlier N(0, 1) default gave an initial loss of about 91 nats instead of ln 65 ≈ 4.17 because the tied output projection produced very large logits.

In [ ]:
class TransformerLM(nn.Module):
    def __init__(self, vocab_size=65, d_model=128, heads=4, layers=4, d_ff=512, dropout=.1, context=128):
        super().__init__(); self.context = context
        self.token = nn.Embedding(vocab_size, d_model)
        nn.init.normal_(self.token.weight, std=.02)
        self.pos = SinusoidalPositionalEncoding(d_model, context + 1)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(d_model); self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token.weight
    def forward(self, idx, causal=True):          # idx: (B, L) -> logits (B, L, V)
        B, L = idx.shape; x = self.pos(self.token(idx))
        mask = torch.tril(torch.ones(L, L, device=idx.device, dtype=torch.bool))[None, None] if causal else None
        for layer in self.blocks: x = layer(x, mask)
        return self.lm_head(self.norm(x))
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=500, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            inp = idx[:, -self.context:]; logits = self(inp)[:, -1, :] / max(temperature, 1e-5)
            if top_k:
                vals, inds = torch.topk(logits, min(top_k, logits.size(-1)))
                filt = torch.full_like(logits, float("-inf")); filt.scatter_(1, inds, vals); logits = filt
            probs = torch.softmax(logits, dim=-1); nxt = torch.multinomial(probs, 1)
            idx = torch.cat([idx, nxt], dim=1)
        return idx

@torch.no_grad()
def eval_lm(model, batches):
    # Always evaluated WITH the causal mask (next-character prediction), in nats per character
    model.eval(); tot = 0.0; n = 0
    for x, y in batches:
        logits = model(x.to(DEVICE))
        tot += F.cross_entropy(logits.reshape(-1, logits.size(-1)), y.to(DEVICE).reshape(-1), reduction="sum").item()
        n += y.numel()
    model.train(); return tot / n

def train_lm(heads=4, steps=3000, causal=True, lr=3e-4, eval_every=250, seed=SEED):
    # Constant lr 3e-4, AdamW, batch 64 x context 128, grad-clip 1.0
    set_seed(seed)
    model = TransformerLM(heads=heads).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    curve = {"step": [], "train_loss": [], "val_nats": []}
    window = []; t0 = time.time()
    for step in range(steps):
        model.train()
        xb, yb = get_batch(train_tokens)
        logits = model(xb, causal=causal)
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), yb.reshape(-1))
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        window.append(loss.item())
        if step % eval_every == 0 or step == steps - 1:
            v = eval_lm(model, VAL_EVAL); tr = float(np.mean(window)); window = []
            curve["step"].append(step); curve["train_loss"].append(tr); curve["val_nats"].append(v)
            print(f"  step {step:5d}: train {tr:.3f}  val {v:.3f} nats/char")
    return model, curve, (time.time() - t0) / 60

### 4.2 Default language model (4 heads, 4 layers, d = 128)

In [ ]:
LM_STEPS = 3000
lm, lm_curve, lm_minutes = train_lm(heads=4, steps=LM_STEPS)
lm_val_nats = eval_lm(lm, VAL_EVAL)
lm_val_bits = lm_val_nats / math.log(2)
print(f"Initial train loss (step 0): {lm_curve['train_loss'][0]:.3f}   (ln 65 = {math.log(65):.3f})")
print(f"Validation nats/char: {lm_val_nats:.4f}   bits/char: {lm_val_bits:.4f}   training minutes: {lm_minutes:.1f}")
print("Parameters:", parameter_count(lm))

In [ ]:
plt.plot(lm_curve["step"], lm_curve["train_loss"], label="train (mean of window)")
plt.plot(lm_curve["step"], lm_curve["val_nats"], label="validation (fixed windows)")
plt.xlabel("Training step"); plt.ylabel("Loss (nats/character)"); plt.legend(); plt.tight_layout()
plt.savefig(FIG/"lm_learning_curve.png", dpi=160); plt.show()

### 4.3 Ablations on the language model (same budget: 800 steps each)

Heads = 1, 4, 8 at fixed d = 128, and the no-causal-mask model. The heads=4 run here is trained separately from the 3000-step default model so that all four heads runs share one budget. The no-mask model is trained with full bidirectional attention (`causal=False`) but is **evaluated with the causal mask**, because that is the next-character task.

In [ ]:
ABL_STEPS = 800
ablation = {}
for h in [1, 4, 8]:
    m, c, mins = train_lm(heads=h, steps=ABL_STEPS)
    ablation[f"heads={h}"] = {"model": m, "curve": c, "minutes": mins, "val_nats": eval_lm(m, VAL_EVAL), "params": parameter_count(m)}
m, c, mins = train_lm(heads=4, steps=ABL_STEPS, causal=False)
ablation["no causal mask (heads=4)"] = {"model": m, "curve": c, "minutes": mins, "val_nats": eval_lm(m, VAL_EVAL), "params": parameter_count(m)}
abl_table = pd.DataFrame([{"variant": k, "params": v["params"], "train_steps": ABL_STEPS,
                           "final_train_loss": v["curve"]["train_loss"][-1],
                           "val_nats": v["val_nats"], "val_bits": v["val_nats"] / math.log(2),
                           "minutes": round(v["minutes"], 2)} for k, v in ablation.items()])
display(abl_table)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
names = list(ablation.keys())
ax[0].bar(range(len(names)), [ablation[k]["val_nats"] / math.log(2) for k in names])
ax[0].set_xticks(range(len(names))); ax[0].set_xticklabels(names, rotation=20, ha="right")
ax[0].set_ylabel("validation bits/char (lower is better)"); ax[0].set_title("Ablation: validation bits/char")
for k in ["heads=4", "no causal mask (heads=4)"]:
    ax[1].plot(ablation[k]["curve"]["step"], ablation[k]["curve"]["train_loss"], label=k)
ax[1].set_xlabel("Training step"); ax[1].set_ylabel("train loss (nats/char)"); ax[1].legend(); ax[1].set_title("Causal vs no mask: training loss")
plt.tight_layout(); plt.savefig(FIG/"lm_ablations.png", dpi=160); plt.show()

### 4.4 Temperature sampling — 500 characters each (top-k = 40)

In [ ]:
set_seed(SEED)
start = torch.tensor([[stoi[" "]]], device=DEVICE)
samples = {}
for temp in [0.2, 0.8, 1.5]:
    ids = lm.generate(start, max_new_tokens=500, temperature=temp, top_k=40)[0].tolist()
    samples[temp] = "".join(itos[i] for i in ids)
    print(f"\n--- temperature={temp} ---\n{samples[temp]}")
with open("generated_samples.txt", "w") as f:
    for temp, s in samples.items(): f.write(f"Temperature {temp} (top-k 40, 500 characters)\n{s}\n\n")

In [ ]:
set_seed(SEED)
nomask = ablation["no causal mask (heads=4)"]["model"]
ids = nomask.generate(start, max_new_tokens=500, temperature=0.8, top_k=40)[0].tolist()
nomask_sample = "".join(itos[i] for i in ids)
print("No-mask model, temperature 0.8 (generation uses the causal mask it never saw in training):\n", nomask_sample)
with open("generated_samples_nomask.txt", "w") as f: f.write("No causal mask in training, temperature 0.8 (top-k 40)\n" + nomask_sample + "\n")

## 5. Attention maps, context-length timing

### 5.1 Attention heatmaps (default model, layer 1, all 4 heads)

In [ ]:
# Visualize attention weights for an input sequence (head x query position x key position).
def plot_attention(model,encoded_sentence):
    model.eval(); idx=torch.tensor([[stoi[c] for c in encoded_sentence]],device=DEVICE)
    x=model.pos(model.token(idx)); L=idx.size(1)
    mask=torch.tril(torch.ones(L,L,device=DEVICE,dtype=torch.bool))[None,None]
    weights=[]
    with torch.no_grad():
        for layer in model.blocks:
            _,w=layer.attn(layer.norm1(x),mask,return_attention=True)
            weights.append(w[0].cpu().numpy())
            x=layer(x,mask)
    # Plot first layer's heads
    heads=weights[0]
    fig,axes=plt.subplots(1,heads.shape[0],figsize=(4*heads.shape[0],4))
    for h,ax in enumerate(np.atleast_1d(axes)):
        ax.imshow(heads[h],origin="lower",aspect="auto"); ax.set_title(f"Head {h+1}")
        ax.set_xlabel("Key position"); ax.set_ylabel("Query position")
    plt.tight_layout(); plt.savefig(FIG/"attention_heads.png",dpi=160); plt.show()
plot_attention(lm,"To be, or not to be")

<!--PATCH:heatmap-->

### 5.2 Context-length timing (O(n²·d) attention)

Sec/step is the average of 20 training steps at batch 64 after one untimed warm-up step. Peak memory is read from CUDA only; this machine runs on CPU, so memory is reported as n/a. The attention score matrix is n×n per head, so its compute and memory grow with n²·d; the feed-forward and projection terms grow with n·d².

In [ ]:
def timed_train_step(model, context, steps=20):
    model.train(); opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
    xb, yb = get_batch(train_tokens, context=context)           # warm-up step, not timed
    loss = F.cross_entropy(model(xb).reshape(-1, len(chars)), yb.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step()
    if DEVICE.type == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    t = time.perf_counter()
    for _ in range(steps):
        xb, yb = get_batch(train_tokens, context=context); opt.zero_grad()
        logits = model(xb); loss = F.cross_entropy(logits.reshape(-1, len(chars)), yb.reshape(-1))
        loss.backward(); opt.step()
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    elapsed = time.perf_counter() - t
    peak = torch.cuda.max_memory_allocated() / 1024**2 if DEVICE.type == "cuda" else None
    return {"sec_per_step": elapsed / steps, "parameters": parameter_count(model), "peak_memory_mb": peak}

TIMING = []
for ctx in [64, 128, 256, 512]:
    set_seed()
    m = TransformerLM(context=ctx).to(DEVICE)
    TIMING.append({"context": ctx, **timed_train_step(m, ctx, steps=20)})
    del m
timing_df = pd.DataFrame(TIMING)
display(timing_df)
plt.plot(timing_df["context"], timing_df["sec_per_step"], "o-")
plt.xscale("log", base=2); plt.xlabel("Context length (tokens)"); plt.ylabel("sec / training step (batch 64)")
plt.title("Training time vs context (CPU)"); plt.tight_layout(); plt.savefig(FIG/"context_timing.png", dpi=160); plt.show()

## 6. Save artifacts and results

In [ ]:
import pandas as pd
# Config is derived from the trained model, not hard-coded
cfg = {"context": CONTEXT, "d_model": lm.token.embedding_dim, "heads": lm.blocks[0].attn.num_heads,
       "layers": len(lm.blocks), "d_ff": lm.blocks[0].ff[0].out_features, "dropout": lm.blocks[0].attn.dropout,
       "steps": LM_STEPS, "lr": 3e-4, "seed": SEED, "causal": True}
assert cfg["layers"] == 4 and cfg["heads"] == 4
torch.save({"model_state_dict": lm.state_dict(), "vocab": chars, "config": cfg}, "transformer_lm.pt")
torch.save(HAR_RUNS["sinusoidal"]["model"].state_dict(), "har_transformer.pt")

har_test = HAR_RUNS["sinusoidal"]["test"]
pd.DataFrame([
    {"experiment": "HAR Transformer", "metric": "test accuracy", "value": har_test["accuracy"]},
    {"experiment": "HAR Transformer", "metric": "test macro-F1", "value": har_test["macro_f1"]},
    {"experiment": "Tiny Shakespeare", "metric": "validation nats/char", "value": lm_val_nats},
    {"experiment": "Tiny Shakespeare", "metric": "validation bits/char", "value": lm_val_bits},
]).to_csv("problem2_results.csv", index=False)

rows = []
def add(model, seq_len, test_acc=None, macro_f1=None, params=None, sec=None, val_nats=None, bits=None, notes=""):
    rows.append({"problem": 2, "model": model, "seed": SEED, "seq_len": seq_len, "test_acc": test_acc,
                 "macro_f1": macro_f1, "test_mse": None, "params": params, "sec_per_epoch": sec,
                 "val_loss_nats": val_nats, "bits_per_char": bits, "notes": notes})
for p, r in HAR_RUNS.items():
    add(f"HAR Transformer (pos={p})", 128, r["test"]["accuracy"], r["test"]["macro_f1"], r["params"],
        round(r["sec_per_epoch"], 2), notes=f"15 epochs; val_acc={r['val']['accuracy']:.4f}; official test split")
add("Char Transformer LM (default, 4 heads, 4 layers)", CONTEXT, params=parameter_count(lm),
    val_nats=lm_val_nats, bits=lm_val_bits,
    notes=f"{LM_STEPS} steps; constant lr 3e-4; 100x64 fixed val windows; {lm_minutes:.1f} min CPU")
for k, v in ablation.items():
    add(f"Char Transformer LM ablation: {k}", CONTEXT, params=v["params"], val_nats=v["val_nats"],
        bits=v["val_nats"] / math.log(2), notes=f"{ABL_STEPS} steps (same budget for all ablation runs)")
for t in TIMING:
    add("Timing: Char Transformer LM", t["context"], notes=f"sec_per_step={t['sec_per_step']:.4f}; batch 64; "
        f"peak_memory_mb={'n/a (CPU)' if t['peak_memory_mb'] is None else round(t['peak_memory_mb'], 1)}")
pd.DataFrame(rows).to_csv("results_rows.csv", index=False)
print("Saved transformer_lm.pt, har_transformer.pt, problem2_results.csv, results_rows.csv, generated_samples*.txt")
display(pd.read_csv("problem2_results.csv"))

## 7. Discussion

### 7.1 Effect of Scaling by \(\sqrt{d_k}\)

In scaled dot-product attention, the query-key similarity scores are divided by \(\sqrt{d_k}\). Without this scaling, the dot products can become large when the key dimension increases. Large scores cause the softmax distribution to become overly concentrated, resulting in smaller gradients.

Scaling helps maintain a more stable softmax distribution and improves the learning behavior of attention.

### 7.2 Attention head interpretations

<!--PATCH:heads-discussion-->

<!--PATCH:discussion-->